# South Carolina 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for South Carolina, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals). Note, there are 17 missing counties in the general election data.

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `dem_primary_total`, `rep_general_total`, `dem_general_total`, `lib_general_total`, `grn_general_total`, `con_general_total`, `pet_general_total`, `non_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [3]:
import re 
import pandas as pd
import numpy as numpy
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

In [1]:
# SC 2008 dataset path
PRIMARY_DEM_PATH = r"../../data/raw/2008/SC/20080126__sc__democratic__primary__president__county.csv"
PRIMARY_REP_PATH = r"../../data/raw/2008/SC/20080119__sc__republican__primary__president__county.csv"
GENERAL_PATH     = r"../../data/raw/2008/SC/20081104__sc__general__precinct.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/SC/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [4]:
# Load primary data
primary_dem = pd.read_csv(PRIMARY_DEM_PATH)
primary_rep = pd.read_csv(PRIMARY_REP_PATH)

# Join the two sub-dataframe into one
primary_df = pd.concat([primary_dem, primary_rep], ignore_index=True, sort=False)
primary_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,ABBEVILLE,President,NaN,DEM,Joe Biden,6
1,ABBEVILLE,President,NaN,DEM,Hillary Clinton,738
2,ABBEVILLE,President,NaN,DEM,Chris Dodd,1
3,ABBEVILLE,President,NaN,DEM,John Edwards,802
4,ABBEVILLE,President,NaN,DEM,Mike Gravel,2
5,ABBEVILLE,President,NaN,DEM,Dennis Kucinich,2
6,ABBEVILLE,President,NaN,DEM,Barack Obama,2006
7,ABBEVILLE,President,NaN,DEM,Bill Richardson,1
8,AIKEN,President,NaN,DEM,Joe Biden,18
9,AIKEN,President,NaN,DEM,Hillary Clinton,4901


In [5]:
# Different values in 'office' column
primary_df["office"].value_counts()

office
President    893
Name: count, dtype: int64

In [6]:
# Number of missing values in each column
primary_df.isna().sum()

county         0
office         0
district     893
party          0
candidate      0
votes          0
dtype: int64

In [7]:
# Drop the "office" column since it has only "President" value
# Also, drop the district column since it's all missing values
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,ABBEVILLE,DEM,Joe Biden,6
1,ABBEVILLE,DEM,Hillary Clinton,738
2,ABBEVILLE,DEM,Chris Dodd,1
3,ABBEVILLE,DEM,John Edwards,802
4,ABBEVILLE,DEM,Mike Gravel,2
5,ABBEVILLE,DEM,Dennis Kucinich,2
6,ABBEVILLE,DEM,Barack Obama,2006
7,ABBEVILLE,DEM,Bill Richardson,1
8,AIKEN,DEM,Joe Biden,18
9,AIKEN,DEM,Hillary Clinton,4901


In [8]:
# Different candidates in primary_df
primary_df["candidate"].value_counts()

candidate
Joe Biden          47
Cap Fendig         47
Tom Tancredo       47
Mitt Romney        47
Ron Paul           47
John McCain        47
Duncan Hunter      47
Mike Huckabee      47
Rudy Giuliani      47
John Cox           47
Hillary Clinton    47
Hugh Cort          47
Bill Richardson    47
Barack Obama       47
Dennis Kucinich    47
Mike Gravel        47
John Edwards       47
Chris Dodd         47
Fred Thompson      47
Name: count, dtype: int64

In [9]:
# List out all the parties in the general election data
primary_df["party"].value_counts()

party
REP    517
DEM    376
Name: count, dtype: int64

In [10]:
# Data type of each column in primary_df
primary_df.dtypes

county       object
party        object
candidate    object
votes         int64
dtype: object

In [11]:
# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,ABBEVILLE,DEM,Joe Biden,6
1,ABBEVILLE,DEM,Hillary Clinton,738
2,ABBEVILLE,DEM,Chris Dodd,1
3,ABBEVILLE,DEM,John Edwards,802
4,ABBEVILLE,DEM,Mike Gravel,2
5,ABBEVILLE,DEM,Dennis Kucinich,2
6,ABBEVILLE,DEM,Barack Obama,2006
7,ABBEVILLE,DEM,Bill Richardson,1
8,AIKEN,DEM,Joe Biden,18
9,AIKEN,DEM,Hillary Clinton,4901


In [12]:
# Shape after preprocessing
primary_df.shape

(893, 4)

### b. General Election Dataset

In [13]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,votes
0,Abbeville,Abbeville No. 2,Straight Party,NaN,GRN,Green,2
1,Abbeville,Abbeville No. 3,Straight Party,NaN,GRN,Green,1
2,Abbeville,Abbeville No. 4,Straight Party,NaN,GRN,Green,2
3,Abbeville,Antreville,Straight Party,NaN,GRN,Green,4
4,Abbeville,Broadmouth,Straight Party,NaN,GRN,Green,3
5,Abbeville,Calhoun Falls,Straight Party,NaN,GRN,Green,6
6,Abbeville,Cold Springs,Straight Party,NaN,GRN,Green,1
7,Abbeville,Donalds,Straight Party,NaN,GRN,Green,2
8,Abbeville,Due West,Straight Party,NaN,GRN,Green,3
9,Abbeville,Halls Store,Straight Party,NaN,GRN,Green,2


In [14]:
# Different values in 'office' column
general_df["office"].value_counts()

office
President                                               7286
Straight Party                                          6354
U.S. Senate                                             3553
Amendment No. 3                                         2799
Amendment No. 2                                         2799
                                                        ... 
City Council District 12                                   4
Watershed District Commissioners                           3
City Council District 14                                   3
State House of Representatives District 7                  3
Oolenoy Watershed District Commissioners District 24       3
Name: count, Length: 375, dtype: int64

In [15]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,votes
67,Abbeville,Abbeville No. 1,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,1
68,Abbeville,Abbeville No. 3,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,2
69,Abbeville,Antreville,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,3
70,Abbeville,Broadmouth,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,1
71,Abbeville,Calhoun Falls,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,5
72,Abbeville,Cold Springs,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,2
73,Abbeville,Due West,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,1
74,Abbeville,Halls Store,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,1
75,Abbeville,Keowee,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,3
76,Abbeville,Lowndesville,President,NaN,GRN,Cynthia McKinney / Rosa Clemente,1


In [16]:
# General data shape when only considering President/VicePresident
general_df.shape

(7286, 7)

In [17]:
# Number of missing values in each column
general_df.isna().sum()

county          0
precinct        0
office          0
district     7286
party           0
candidate       0
votes           0
dtype: int64

In [19]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,precinct,party,candidate,votes
0,Abbeville,Abbeville No. 1,GRN,Cynthia McKinney / Rosa Clemente,1
1,Abbeville,Abbeville No. 3,GRN,Cynthia McKinney / Rosa Clemente,2
2,Abbeville,Antreville,GRN,Cynthia McKinney / Rosa Clemente,3
3,Abbeville,Broadmouth,GRN,Cynthia McKinney / Rosa Clemente,1
4,Abbeville,Calhoun Falls,GRN,Cynthia McKinney / Rosa Clemente,5
5,Abbeville,Cold Springs,GRN,Cynthia McKinney / Rosa Clemente,2
6,Abbeville,Due West,GRN,Cynthia McKinney / Rosa Clemente,1
7,Abbeville,Halls Store,GRN,Cynthia McKinney / Rosa Clemente,1
8,Abbeville,Keowee,GRN,Cynthia McKinney / Rosa Clemente,3
9,Abbeville,Lowndesville,GRN,Cynthia McKinney / Rosa Clemente,1


In [20]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
John McCain / Sarah Palin           1464
Barack Obama / Joe Biden            1388
Bob Barr / Wayne A Root             1195
Cynthia McKinney / Rosa Clemente    1107
Chuck Baldwin / Darrell Castle      1075
Ralph Nader / Matt Gonzalez         1057
Name: count, dtype: int64

Now, each row’s candidate value now contains two names: presidential first, vice-presidential second, which is separated by a backslash ("/"). We’ll split on the backslash and retain only the presidential name.

In [21]:
# Keep only the presidential candidate in the "candidate" column
general_df["candidate"] = (
    general_df["candidate"]
      .str.split(r"(?i)\s*(?:andf|and|&|/|/)\s*", n=1, expand=True)[0]
      .str.strip()
)

# Candidates in general_df
general_df["candidate"].value_counts()

candidate
John McCain         1464
Barack Obama        1388
Bob Barr            1195
Cynthia McKinney    1107
Chuck Baldwin       1075
Ralph Nader         1057
Name: count, dtype: int64

Given that the general election data is given on precinct-level, we will compute county-level vote totals by grouping on precinct and summing the precinct votes.

In [24]:
# Calculate county-level votes
general_df = (
    general_df.groupby(["county", "candidate", "party"], as_index=False)["votes"].sum()
)

general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Abbeville,Barack Obama,DEM,4593
1,Abbeville,Bob Barr,LIB,25
2,Abbeville,Chuck Baldwin,CON,55
3,Abbeville,Cynthia McKinney,GRN,26
4,Abbeville,John McCain,REP,6264
5,Abbeville,Ralph Nader,PET,38
6,Aiken,Bob Barr,LIB,158
7,Aiken,Cynthia McKinney,GRN,160
8,Aiken,John McCain,REP,42849
9,Fairfield,Barack Obama,DEM,7591


In [25]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
LIB    29
GRN    29
REP    29
DEM    28
CON    28
PET    27
NON     1
Name: count, dtype: int64

I just want to have a quick look at the observations where `party == "NON"`.

In [26]:
# Row where "party" == "NON"
general_df[general_df["party"] == "NON"]

,county,candidate,party,votes
32,Greenville,Ralph Nader,NON,526


In [27]:
# Data type of each column in general_df
general_df.dtypes

county       object
candidate    object
party        object
votes         int64
dtype: object

In [28]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Abbeville,Barack Obama,DEM,4593
1,Abbeville,Bob Barr,LIB,25
2,Abbeville,Chuck Baldwin,CON,55
3,Abbeville,Cynthia McKinney,GRN,26
4,Abbeville,John McCain,REP,6264
5,Abbeville,Ralph Nader,PET,38
6,Aiken,Bob Barr,LIB,158
7,Aiken,Cynthia McKinney,GRN,160
8,Aiken,John McCain,REP,42849
9,Fairfield,Barack Obama,DEM,7591


In [29]:
# Shape after preprocessing
general_df.shape

(171, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [30]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names by lowering the three-letter abbreviations
    """
    return(s.str.lower())

In [31]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [32]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [33]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_CORT,pri_rep_COX,pri_rep_FENDIG,pri_rep_GIULIANI,pri_rep_HUCKABEE,pri_rep_HUNTER,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON
0,ABBEVILLE,6,738,1,802,2,2,2006,1,1,0,0,11,907,3,541,104,207,0,359
1,AIKEN,18,4901,11,1866,10,15,7768,29,3,2,2,435,5901,36,6379,822,2639,2,2081
2,ALLENDALE,4,392,0,92,1,4,951,4,0,2,0,7,70,0,144,12,27,0,26
3,ANDERSON,11,5485,5,5603,7,10,6315,16,1,3,1,194,8474,43,5996,769,2594,3,2983
4,BAMBERG,11,751,3,222,3,9,1720,5,0,0,0,9,265,1,341,25,76,0,71
5,BARNWELL,5,820,2,273,1,3,1632,4,0,0,0,32,611,1,700,46,104,0,111
6,BEAUFORT,39,5115,4,2239,5,15,9550,27,2,4,2,1122,3211,12,8249,358,5061,2,1623
7,BERKELEY,23,3880,14,2676,8,18,9869,38,2,4,1,315,3991,46,5840,525,2248,4,2228
8,CALHOUN,4,699,0,413,2,5,1607,4,0,1,0,18,482,1,503,54,164,2,219
9,CHARLESTON,67,11287,18,6438,39,56,30073,66,6,6,4,1259,5584,65,15188,1457,6494,3,4179


In [34]:
# Primary dataframe shape after pivot
primary_pivot.shape

(47, 20)

There is something not quite right with the shape of the primary data after pivoting. Given that there are only 46 counties in South Carolina, we expect that the pivoted dataframe has up to 46 rows. But here, we have 47. Thus, we need further investigation on this `county` variable.

In [43]:
# List of all counties in `primary_pivot`
primary_pivot["county"].values

array(['ABBEVILLE', 'AIKEN', 'ALLENDALE', 'ANDERSON', 'BAMBERG',
       'BARNWELL', 'BEAUFORT', 'BERKELEY', 'CALHOUN', 'CHARLESTON',
       'CHEROKEE', 'CHESTER', 'CHESTERFIELD', 'CLARENDON', 'COLLETON',
       'DARLINGTON', 'DILLON', 'DORCHESTER', 'EDGEFIELD', 'FAIRFIELD',
       'FLORENCE', 'GEORGETOWN', 'GREENVILLE', 'GREENWOOD', 'HAMPTON',
       'HORRY', 'JASPER', 'KERSHAW', 'LANCASTER', 'LAURENS', 'LEE',
       'LEXINGTON', 'MARION', 'MARLBORO', 'MCCORMICK', 'NEWBERRY',
       'OCONEE', 'ORANGEBURG', 'PICKENS', 'RICHLAND', 'SALUDA',
       'SPARTANBURG', 'SUMTER', 'Totals', 'UNION', 'WILLIAMSBURG', 'YORK'],
      dtype=object)

This shows that there is a total row that we missed. Thus, we can just dropped this row and we will be good to go.

In [44]:
# Drop the total row in `primary_pvit`
primary_pivot = primary_pivot[primary_pivot["county"] != "Totals"]
primary_pivot.shape

(46, 20)

In [35]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_con_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_non_NADER,gen_pet_NADER,gen_rep_MCCAIN
0,Abbeville,55,4593,26,25,0,38,6264
1,Aiken,0,0,160,158,0,0,42849
2,Fairfield,35,7591,32,27,0,22,3912
3,Florence,125,28012,109,151,0,115,29861
4,Georgetown,61,14199,83,89,0,68,15790
5,Greenville,1566,70886,423,893,526,0,116363
6,Greenwood,97,12348,58,100,0,69,16995
7,Hampton,18,5816,28,34,0,15,3439
8,Horry,273,38879,269,367,0,401,64609
9,Jasper,17,5389,35,32,0,16,3365


Notice that the county names in `general_pivot` only capitalize the first letter, while such names in `primary_pivot` capitalize the whole word. Thus, for the convenience of merging, we can just capitalize the whole county name in general.

In [39]:
# CAPITALIZE county name
general_pivot["county"] = general_pivot["county"].str.upper()
general_pivot.head(DISPLAY_ROWS)

,county,gen_con_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_non_NADER,gen_pet_NADER,gen_rep_MCCAIN
0,ABBEVILLE,55,4593,26,25,0,38,6264
1,AIKEN,0,0,160,158,0,0,42849
2,FAIRFIELD,35,7591,32,27,0,22,3912
3,FLORENCE,125,28012,109,151,0,115,29861
4,GEORGETOWN,61,14199,83,89,0,68,15790
5,GREENVILLE,1566,70886,423,893,526,0,116363
6,GREENWOOD,97,12348,58,100,0,69,16995
7,HAMPTON,18,5816,28,34,0,15,3439
8,HORRY,273,38879,269,367,0,401,64609
9,JASPER,17,5389,35,32,0,16,3365


In [40]:
# General dataframe shape after pivot
general_pivot.shape

(29, 8)

Here, for the pivoted general dataframe, we only have 29 rows. Just to make sure that we didn't include the total rows in `general_pivot`, we first check for that and drop if exists. Then, we will figure out which counties are missing from the general dataset.

In [45]:
# List of all counties in `general_pivot`
general_pivot["county"].values

array(['ABBEVILLE', 'AIKEN', 'FAIRFIELD', 'FLORENCE', 'GEORGETOWN',
       'GREENVILLE', 'GREENWOOD', 'HAMPTON', 'HORRY', 'JASPER', 'KERSHAW',
       'LANCASTER', 'LAURENS', 'LEE', 'LEXINGTON', 'MARION', 'MARLBORO',
       'MCCORMICK', 'NEWBERRY', 'OCONEE', 'ORANGEBURG', 'PICKENS',
       'RICHLAND', 'SALUDA', 'SPARTANBURG', 'SUMTER', 'UNION',
       'WILLIAMSBURG', 'YORK'], dtype=object)

In [51]:
# Missing counties in general_pivot
missing_counties = [c for c in primary_pivot["county"].values if c not in general_pivot["county"].values]
missing_counties

['ALLENDALE',
 'ANDERSON',
 'BAMBERG',
 'BARNWELL',
 'BEAUFORT',
 'BERKELEY',
 'CALHOUN',
 'CHARLESTON',
 'CHEROKEE',
 'CHESTER',
 'CHESTERFIELD',
 'CLARENDON',
 'COLLETON',
 'DARLINGTON',
 'DILLON',
 'DORCHESTER',
 'EDGEFIELD']

## 4. Merge Dataframes

Given the missing counties in the general dataframe, the merged dataframe should likewise lack 17 counties, as shown in `general_pivot`.

In [52]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_CORT,...,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON,gen_con_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_non_NADER,gen_pet_NADER,gen_rep_MCCAIN
0,ABBEVILLE,6,738,1,802,2,2,2006,1,1,...,207,0,359,55,4593,26,25,0,38,6264
1,AIKEN,18,4901,11,1866,10,15,7768,29,3,...,2639,2,2081,0,0,160,158,0,0,42849
2,FAIRFIELD,6,996,2,537,2,7,3410,9,1,...,224,1,259,35,7591,32,27,0,22,3912
3,FLORENCE,17,3700,7,2222,3,16,10768,19,1,...,1499,4,1548,125,28012,109,151,0,115,29861
4,GEORGETOWN,11,1951,6,1573,4,14,5334,16,1,...,1352,2,911,61,14199,83,89,0,68,15790
5,GREENVILLE,56,11918,13,9047,19,49,21532,43,11,...,10231,12,12495,1566,70886,423,893,526,0,116363
6,GREENWOOD,12,1528,4,1436,5,8,4373,22,0,...,854,2,1065,97,12348,58,100,0,69,16995
7,HAMPTON,7,726,4,362,2,2,2232,6,0,...,59,0,65,18,5816,28,34,0,15,3439
8,HORRY,20,9983,21,7249,14,29,8541,23,3,...,4737,10,3608,273,38879,269,367,0,401,64609
9,JASPER,7,577,2,190,4,1,2285,10,0,...,142,1,83,17,5389,35,32,0,16,3365


In [53]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_CORT,pri_rep_COX,...,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON,gen_con_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_non_NADER,gen_pet_NADER,gen_rep_MCCAIN
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,...,29.00000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,15.724138,3440.758621,5.620690,2279.000000,5.689655,12.551724,6941.689655,15.793103,2.448276,1.965517,...,1612.62069,2.551724,1833.137931,173.379310,19530.793103,106.724138,169.068966,18.137931,96.344828,25197.448276
std,15.702358,3763.311855,5.066795,2342.447331,5.189987,13.642335,8152.581748,14.465178,5.735766,2.291019,...,2337.17501,3.123485,2661.572464,307.378015,22834.677595,101.797031,205.719110,97.675748,114.122514,27940.911626
min,3.000000,382.000000,0.000000,186.000000,1.000000,1.000000,1195.000000,1.000000,0.000000,0.000000,...,59.00000,0.000000,65.000000,0.000000,0.000000,15.000000,9.000000,0.000000,0.000000,2437.000000
25%,6.000000,955.000000,2.000000,760.000000,2.000000,4.000000,2285.000000,7.000000,0.000000,1.000000,...,201.00000,0.000000,259.000000,34.000000,5960.000000,35.000000,34.000000,0.000000,21.000000,5191.000000
50%,11.000000,1951.000000,4.000000,1505.000000,4.000000,8.000000,4353.000000,12.000000,1.000000,1.000000,...,520.00000,1.000000,979.000000,65.000000,11226.000000,68.000000,100.000000,0.000000,55.000000,15790.000000
75%,17.000000,4581.000000,9.000000,2815.000000,8.000000,16.000000,9020.000000,21.000000,3.000000,2.000000,...,1499.00000,4.000000,2081.000000,132.000000,27263.000000,126.000000,158.000000,0.000000,115.000000,32552.000000
max,70.000000,14888.000000,21.000000,9047.000000,19.000000,58.000000,42146.000000,72.000000,30.000000,9.000000,...,10231.00000,12.000000,12495.000000,1566.000000,105656.000000,423.000000,893.000000,526.000000,401.000000,116363.000000


Now, we will add party totals columns: 

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    * `dem_primary_total` = sum of all `pri_dem_*` columns

- General totals:
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `lib_general_total` = sum of all `gen_lib_*` columns
    * `grn_general_total` = sum of all `gen_grn_*` columns
    * `con_general_total` = sum of all `con_grn_*` columns
    * `pet_general_total` = sum of all `pet_grn_*` columns
    * `non_general_total` = sum of all `pet_non_*` columns

In [54]:
# Add party totals for primary election
rep_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_dem_")]

merged_df["rep_primary_total"] = merged_df[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
merged_df["dem_primary_total"] = merged_df[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0

In [55]:
# Add party totals for general election
rep_general_cols   = [c for c in merged_df.columns if c.startswith("gen_rep_")]
dem_general_cols   = [c for c in merged_df.columns if c.startswith("gen_dem_")]
lib_general_cols   = [c for c in merged_df.columns if c.startswith("gen_lib_")]
grn_general_cols   = [c for c in merged_df.columns if c.startswith("gen_grn_")]
con_general_cols   = [c for c in merged_df.columns if c.startswith("gen_con_")]
pet_general_cols   = [c for c in merged_df.columns if c.startswith("gen_pet_")]
non_general_cols   = [c for c in merged_df.columns if c.startswith("gen_non_")]

merged_df["rep_general_total"] = merged_df[rep_general_cols].sum(axis=1) if rep_general_cols else 0
merged_df["dem_general_total"] = merged_df[dem_general_cols].sum(axis=1) if dem_general_cols else 0
merged_df["lib_general_total"] = merged_df[lib_general_cols].sum(axis=1) if lib_general_cols else 0
merged_df["grn_general_total"] = merged_df[grn_general_cols].sum(axis=1) if grn_general_cols else 0
merged_df["con_general_total"] = merged_df[con_general_cols].sum(axis=1) if con_general_cols else 0
merged_df["pet_general_total"] = merged_df[pet_general_cols].sum(axis=1) if pet_general_cols else 0
merged_df["non_general_total"] = merged_df[non_general_cols].sum(axis=1) if non_general_cols else 0

In [56]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned dataframe:")
merged_df.columns

Final columns in the cleaned dataframe:


Index(['county', 'pri_dem_BIDEN', 'pri_dem_CLINTON', 'pri_dem_DODD',
       'pri_dem_EDWARDS', 'pri_dem_GRAVEL', 'pri_dem_KUCINICH',
       'pri_dem_OBAMA', 'pri_dem_RICHARDSON', 'pri_rep_CORT', 'pri_rep_COX',
       'pri_rep_FENDIG', 'pri_rep_GIULIANI', 'pri_rep_HUCKABEE',
       'pri_rep_HUNTER', 'pri_rep_MCCAIN', 'pri_rep_PAUL', 'pri_rep_ROMNEY',
       'pri_rep_TANCREDO', 'pri_rep_THOMPSON', 'gen_con_BALDWIN',
       'gen_dem_OBAMA', 'gen_grn_MCKINNEY', 'gen_lib_BARR', 'gen_non_NADER',
       'gen_pet_NADER', 'gen_rep_MCCAIN', 'rep_primary_total',
       'dem_primary_total', 'rep_general_total', 'dem_general_total',
       'lib_general_total', 'grn_general_total', 'con_general_total',
       'pet_general_total', 'non_general_total'],
      dtype='object')

In [57]:
# Preview merged dataframe with totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_CORT,...,gen_rep_MCCAIN,rep_primary_total,dem_primary_total,rep_general_total,dem_general_total,lib_general_total,grn_general_total,con_general_total,pet_general_total,non_general_total
0,ABBEVILLE,6,738,1,802,2,2,2006,1,1,...,6264,2133,3558,6264,4593,25,26,55,38,0
1,AIKEN,18,4901,11,1866,10,15,7768,29,3,...,42849,18302,14618,42849,0,158,160,0,0,0
2,FAIRFIELD,6,996,2,537,2,7,3410,9,1,...,3912,1503,4969,3912,7591,27,32,35,22,0
3,FLORENCE,17,3700,7,2222,3,16,10768,19,1,...,29861,11276,16752,29861,28012,151,109,125,115,0
4,GEORGETOWN,11,1951,6,1573,4,14,5334,16,1,...,15790,6645,8909,15790,14199,89,83,61,68,0
5,GREENVILLE,56,11918,13,9047,19,49,21532,43,11,...,116363,59038,42677,116363,70886,893,423,1566,0,526
6,GREENWOOD,12,1528,4,1436,5,8,4373,22,0,...,16995,6862,7388,16995,12348,100,58,97,69,0
7,HAMPTON,7,726,4,362,2,2,2232,6,0,...,3439,851,3341,3439,5816,34,28,18,15,0
8,HORRY,20,9983,21,7249,14,29,8541,23,3,...,64609,25585,25880,64609,38879,367,269,273,401,0
9,JASPER,7,577,2,190,4,1,2285,10,0,...,3365,1064,3076,3365,5389,32,35,17,16,0


Now, we save the cleaned dataframe into the processed directory.

In [58]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "SC.csv", index=False)